[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/087.%20The%20Hub-and-Spoke%20Network%20Design%20Problem/P87-Tier-9_executed.ipynb)

---



# 87. The Hub-and-Spoke Network Design Problem

## Tier 9: QUBO Modeling and the QAOA Workflow

This notebook formulates a simplified hub-and-spoke design problem as a quadratic unconstrained binary optimization (QUBO) model. It then samples binary strings with a classical random sampler to illustrate how a QUBO objective can be evaluated and inspected.

**Important scope note:** The code does not implement QAOA, a quantum circuit, or a quantum device. The sampler draws random bit strings; its output is not evidence of quantum behavior, quantum advantage, or QAOA convergence. The QAOA terms below describe the intended hybrid workflow and the role a genuine quantum circuit would play.

### Goals and workflow

The model chooses which nodes to open as hubs and assigns each node to exactly one selected hub. The workflow is:

1. Generate a 25-node illustrative network with demands, distances, hub-opening costs, and capacities.
2. Express hub-opening and assignment decisions as binary variables.
3. Convert the cost objective and selected constraints into a QUBO energy function.
4. Inspect the QUBO and sample it with a clearly labeled classical random baseline.
5. Decode the best sampled bit string and check whether it satisfies the modeled constraints.

A QAOA implementation would use the QUBO coefficients to define a problem Hamiltonian, alternate it with a mixer Hamiltonian at depth $p$, and use a classical optimizer to tune circuit parameters. That circuit-and-optimizer computation is not performed here.

### Modeling assumptions and limitations

- The network has directed demand: demand from node $j$ to node $i$ is modeled separately from demand in the reverse direction.
- Transportation cost is demand multiplied by great-circle distance. The model does not include a separate hub-to-hub transfer leg.
- A node may be assigned to itself, at zero transportation distance, but it must still be selected as a hub for that assignment to be allowed.
- The model limits the number of hubs and requires one selected hub assignment per node. Capacity limits are generated as data but are not included as QUBO constraints.
- Constraint penalties are finite. A penalized QUBO does not automatically guarantee that its minimum-energy or sampled bit string is feasible; feasibility is checked separately.
- Costs and demands are illustrative synthetic data, not operational estimates. No random seed is set, so generated data and random samples can differ between runs.

### QUBO variables and objective

Let $N=\{0,\ldots,n-1\}$ be the set of nodes. Define $x_i\in\{0,1\}$ to be 1 when node $i$ is selected as a hub, and $y_{ij}\in\{0,1\}$ to be 1 when node $j$ is assigned to hub $i$. Let $f_i$ be the fixed cost of opening hub $i$, $d_{ij}$ the demand from node $j$ to node $i$, and $\ell_{ij}$ the distance between the two nodes. The operating objective is

$$\min_{x,y}\quad \sum_{i\in N} f_i x_i + \sum_{i\in N}\sum_{j\in N} d_{ij}\ell_{ij}y_{ij}.$$

The constraints are represented by squared penalty terms. For an at-most-$K$ hub limit, introduce nonnegative binary slack bits $s_r$ with weights $2^r$ and impose $\sum_i x_i+\sum_r 2^r s_r=K$. The assignment, hub-count, and assignment-to-open-hub penalties are

$$P\sum_{j\in N}\left(\sum_{i\in N}y_{ij}-1\right)^2
+P\left(\sum_{i\in N}x_i+\sum_r2^r s_r-K\right)^2
+P\sum_{i\in N}\sum_{j\in N}y_{ij}(1-x_i),$$

where $P$ is the penalty coefficient. The slack equation can represent every integer from 0 through $K$ when its binary weights cover that range. For binary variables, $z^2=z$, so expanding these terms gives a quadratic polynomial. The code stores that polynomial as $E(z)=c+q^Tz+z^TQz$, with a symmetric matrix $Q$; its function `formulate_qubo` builds the coefficients and `_calculate_qubo_energy` evaluates them.

The full model has $n+n^2+\lceil\log_2(K+1)\rceil$ binary variables. Its unconstrained binary search space therefore has $2^{n+n^2+\lceil\log_2(K+1)\rceil}$ bit strings. This count describes the mathematical search space, not a number of states efficiently explored by the sampler or by a quantum computer. Dense storage for $Q$ uses quadratic memory in the number of binary variables.

### Reading the results

A low QUBO energy is meaningful as a feasible network cost only when the decoded bit string satisfies the assignment, hub-count, and linking constraints. The notebook reports those checks explicitly. The synthetic capacity values are not part of that feasibility test. Random sampling is a baseline for demonstrating QUBO evaluation; it does not establish that the model is solved optimally or that quantum methods outperform classical methods.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")


@dataclass
class QUBOProblem:
    """Store the coefficients of E(z) = c + q.T @ z + z.T @ Q @ z."""

    linear_terms: np.ndarray
    quadratic_terms: np.ndarray
    constant_term: float
    num_variables: int
    variable_labels: List[str]
    num_hub_variables: int
    num_assignment_variables: int
    slack_indices: List[int]
    slack_weights: List[int]
    max_hubs: int
    num_nodes: int


@dataclass
class QAOACircuit:
    """Hold QAOA settings; this notebook does not execute a quantum circuit."""

    num_qubits: int
    depth: int
    gamma_params: List[float]
    beta_params: List[float]
    measurement_shots: int


@dataclass
class QuantumResult:
    """Represent sampled QUBO bit strings without implying quantum execution."""

    best_solution: np.ndarray
    best_energy: float
    solution_counts: Dict[str, int]
    expectation_values: List[float]
    parameters: Tuple[List[float], List[float]]
    quantum_circuit_depth: int
    execution_time: float
    convergence_history: List[float]
    measurement_shots: int


class QuantumHubSpokeOptimizer:
    """Build a hub-location QUBO and expose its illustrative sampling workflow."""

    def __init__(self, num_nodes: int = 25):
        if not 1 <= num_nodes <= 25:
            raise ValueError("num_nodes must be between 1 and 25 for the supplied coordinates.")

        self.num_nodes = num_nodes
        self.qaoa_depth = 3
        self.measurement_shots = 1000
        self.max_iterations = 50
        self.qubo_penalty = 1000
        self._initialize_quantum_network_data()
        self.quantum_simulator = QuantumCircuitSimulator()
        self.classical_optimizer = ClassicalParameterOptimizer()
        self.performance_tracker = QuantumPerformanceTracker()
        self.quantum_results = []
        self.classical_benchmarks = []

    def _initialize_quantum_network_data(self) -> None:
        """Create the synthetic network inputs used by the QUBO formulation."""
        self.coordinates = self._generate_quantum_coordinates()
        self.distance_matrix = self._calculate_quantum_distance_matrix()
        self.demand_matrix = self._generate_quantum_demand_matrix()
        self.hub_costs = self._generate_quantum_hub_costs()
        self.capacity_limits = self._generate_quantum_capacity_limits()
        self.max_hubs = min(8, self.num_nodes // 3)

    def _generate_quantum_coordinates(self) -> np.ndarray:
        """Return the named locations used in the illustrative network."""
        coordinates = [
            [-74.0, 40.7], [-118.2, 34.1], [-87.6, 41.9], [-122.3, 47.6],
            [-71.1, 42.4], [-95.4, 29.8], [-112.1, 33.5], [-80.2, 25.8],
            [2.3, 48.9], [9.9, 53.6], [-0.1, 51.5], [12.5, 55.7],
            [8.7, 45.1], [2.2, 41.4], [121.5, 31.2], [139.7, 35.7],
            [103.8, 1.3], [114.2, 22.3], [126.9, 37.6], [100.5, 13.8],
            [77.2, 28.6], [-46.6, -23.5], [18.4, -33.9], [31.2, 30.1],
            [151.2, -33.9],
        ]
        return np.asarray(coordinates[:self.num_nodes], dtype=float)

    def _get_node_region(self, node_id: int) -> str:
        """Classify a node by its longitude and latitude."""
        longitude, latitude = self.coordinates[node_id]
        if -125 <= longitude <= -60 and 25 <= latitude <= 50:
            return "north_america"
        if -10 <= longitude <= 40 and 35 <= latitude <= 70:
            return "europe"
        if 60 <= longitude <= 150 and -10 <= latitude <= 50:
            return "asia"
        if -80 <= longitude <= -35 and -55 <= latitude <= 10:
            return "south_america"
        if -20 <= longitude <= 50 and -35 <= latitude <= 40:
            return "africa"
        if 110 <= longitude <= 160 and -45 <= latitude <= -10:
            return "oceania"
        return "other"

    def _calculate_quantum_distance_matrix(self) -> np.ndarray:
        """Calculate pairwise great-circle distances in kilometres."""
        distances = np.zeros((self.num_nodes, self.num_nodes), dtype=float)
        earth_radius_km = 6371.0

        for i in range(self.num_nodes):
            longitude_1, latitude_1 = np.radians(self.coordinates[i])
            for j in range(i + 1, self.num_nodes):
                longitude_2, latitude_2 = np.radians(self.coordinates[j])
                delta_latitude = latitude_2 - latitude_1
                delta_longitude = longitude_2 - longitude_1
                haversine = (
                    np.sin(delta_latitude / 2) ** 2
                    + np.cos(latitude_1)
                    * np.cos(latitude_2)
                    * np.sin(delta_longitude / 2) ** 2
                )
                central_angle = 2 * np.arcsin(np.sqrt(np.clip(haversine, 0, 1)))
                distances[i, j] = earth_radius_km * central_angle
                distances[j, i] = distances[i, j]

        return distances

    def _generate_quantum_demand_matrix(self) -> np.ndarray:
        """Generate directed synthetic demand, adjusted by region and distance."""
        regional_multipliers = {
            "north_america": 1.3,
            "europe": 1.2,
            "asia": 1.4,
            "south_america": 0.7,
            "africa": 0.5,
            "oceania": 0.6,
        }
        regions = [self._get_node_region(i) for i in range(self.num_nodes)]
        demands = np.zeros((self.num_nodes, self.num_nodes), dtype=int)

        for origin in range(self.num_nodes):
            for destination in range(self.num_nodes):
                if origin == destination:
                    continue
                base_demand = random.randint(50, 1500)
                regional_factor = (
                    regional_multipliers.get(regions[origin], 1.0)
                    * regional_multipliers.get(regions[destination], 1.0)
                )
                distance_factor = 1.0 / (
                    1.0 + self.distance_matrix[origin, destination] / 8000.0
                )
                demands[origin, destination] = int(
                    base_demand * regional_factor * distance_factor
                )

        return demands

    def _generate_quantum_hub_costs(self) -> np.ndarray:
        """Generate illustrative fixed hub-opening costs."""
        costs = np.random.uniform(100000, 800000, self.num_nodes)
        for i in range(self.num_nodes):
            region = self._get_node_region(i)
            if region in {"north_america", "europe"}:
                costs[i] *= 1.4
            elif region == "asia":
                costs[i] *= 1.2
            else:
                costs[i] *= 0.6

        for hub in [0, 1, 8, 15, 16]:
            if hub < self.num_nodes:
                costs[hub] *= 1.6
        return costs.astype(int)

    def _generate_quantum_capacity_limits(self) -> np.ndarray:
        """Generate capacities for reference; capacities are not constrained here."""
        capacities = np.random.uniform(20000, 80000, self.num_nodes)
        for i in range(self.num_nodes):
            region = self._get_node_region(i)
            if region in {"north_america", "europe"}:
                capacities[i] *= 1.5
            elif region == "asia":
                capacities[i] *= 1.3
            else:
                capacities[i] *= 0.7
        return capacities.astype(int)

    def formulate_qubo(self, max_hubs: Optional[int] = None) -> QUBOProblem:
        """Build the symmetric-matrix QUBO for assignments and a hub limit."""
        if max_hubs is None:
            max_hubs = self.max_hubs
        if not 0 <= max_hubs <= self.num_nodes:
            raise ValueError("max_hubs must be between zero and the number of nodes.")

        num_hub_vars = self.num_nodes
        num_assignment_vars = self.num_nodes * self.num_nodes
        slack_weights = [2 ** bit for bit in range(max_hubs.bit_length())]
        num_variables = num_hub_vars + num_assignment_vars + len(slack_weights)
        linear_terms = np.zeros(num_variables, dtype=float)
        quadratic_terms = np.zeros((num_variables, num_variables), dtype=float)
        constant_term = 0.0
        variable_labels = [f"x_{i}" for i in range(self.num_nodes)]
        variable_labels.extend(
            f"y_{i}_{j}"
            for i in range(self.num_nodes)
            for j in range(self.num_nodes)
        )
        slack_indices = list(range(num_hub_vars + num_assignment_vars, num_variables))
        variable_labels.extend(f"s_{bit}" for bit in range(len(slack_weights)))

        def assignment_index(hub: int, node: int) -> int:
            return num_hub_vars + hub * self.num_nodes + node

        def add_pair(first: int, second: int, coefficient: float) -> None:
            """Store a polynomial pair coefficient in the symmetric Q matrix."""
            if first == second:
                linear_terms[first] += coefficient
            else:
                quadratic_terms[first, second] += coefficient / 2
                quadratic_terms[second, first] += coefficient / 2

        for i in range(self.num_nodes):
            linear_terms[i] += self.hub_costs[i]

        for hub in range(self.num_nodes):
            for node in range(self.num_nodes):
                # Demand is stored as origin, destination; this assignment cost
                # uses the directed flow from the assigned node to the hub.
                assignment_cost = (
                    self.demand_matrix[node, hub] * self.distance_matrix[node, hub]
                )
                linear_terms[assignment_index(hub, node)] += assignment_cost

        penalty = float(self.qubo_penalty)

        # Each node's assignment indicators must sum to one.
        for node in range(self.num_nodes):
            indices = [
                assignment_index(hub, node) for hub in range(self.num_nodes)
            ]
            constant_term += penalty
            for index in indices:
                linear_terms[index] -= penalty
            for position, first in enumerate(indices):
                for second in indices[position + 1:]:
                    add_pair(first, second, 2 * penalty)

        # Binary slack bits turn the at-most constraint into an equality.
        hub_count_indices = list(range(num_hub_vars)) + slack_indices
        hub_count_weights = [1] * num_hub_vars + slack_weights
        constant_term += penalty * max_hubs ** 2
        for index, weight in zip(hub_count_indices, hub_count_weights):
            linear_terms[index] += penalty * (weight ** 2 - 2 * max_hubs * weight)
        for position, first in enumerate(hub_count_indices):
            for second_position in range(position + 1, len(hub_count_indices)):
                second = hub_count_indices[second_position]
                coefficient = (
                    2 * penalty * hub_count_weights[position]
                    * hub_count_weights[second_position]
                )
                add_pair(first, second, coefficient)

        # An assignment is allowed only when its destination hub is selected.
        for hub in range(self.num_nodes):
            for node in range(self.num_nodes):
                assign_index = assignment_index(hub, node)
                linear_terms[assign_index] += penalty
                add_pair(hub, assign_index, -penalty)

        qubo_problem = QUBOProblem(
            linear_terms=linear_terms,
            quadratic_terms=quadratic_terms,
            constant_term=constant_term,
            num_variables=num_variables,
            variable_labels=variable_labels,
            num_hub_variables=num_hub_vars,
            num_assignment_variables=num_assignment_vars,
            slack_indices=slack_indices,
            slack_weights=slack_weights,
            max_hubs=max_hubs,
            num_nodes=self.num_nodes,
        )

        print("QUBO formulation")
        print(f"Nodes: {self.num_nodes}")
        print(f"Hub limit: {max_hubs}")
        print(f"Binary variables: {num_variables}")
        print(f"  Hub variables: {num_hub_vars}")
        print(f"  Assignment variables: {num_assignment_vars}")
        print(f"  Slack variables: {len(slack_weights)}")
        print(f"Nonzero entries in symmetric Q: {np.count_nonzero(quadratic_terms)}")
        print(f"Constant term: {constant_term:.2f}")
        return qubo_problem

    def _post_process_quantum_result(
        self, quantum_result: QuantumResult, qubo_problem: QUBOProblem
    ) -> Dict[str, object]:
        """Decode one sampled bit string and report its modeled feasibility."""
        solution = quantum_result.best_solution
        num_nodes = qubo_problem.num_nodes
        hubs = [i for i in range(num_nodes) if solution[i] == 1]
        assignments: Dict[int, List[int]] = {}
        assignment_counts = np.zeros(num_nodes, dtype=int)

        for hub in range(num_nodes):
            for node in range(num_nodes):
                index = num_nodes + hub * num_nodes + node
                if solution[index] == 1:
                    assignments.setdefault(node, []).append(hub)
                    assignment_counts[node] += 1

        exactly_one_assignment = bool(np.all(assignment_counts == 1))
        assignments_use_open_hubs = all(
            hub in hubs
            for assigned_hubs in assignments.values()
            for hub in assigned_hubs
        )
        hub_limit_satisfied = len(hubs) <= qubo_problem.max_hubs
        feasible = (
            exactly_one_assignment
            and assignments_use_open_hubs
            and hub_limit_satisfied
        )

        hub_cost = float(np.sum(self.hub_costs[hubs])) if hubs else 0.0
        transportation_cost = 0.0
        for node, assigned_hubs in assignments.items():
            for hub in assigned_hubs:
                transportation_cost += (
                    self.demand_matrix[node, hub] * self.distance_matrix[node, hub]
                )

        return {
            "hub_configuration": hubs,
            "spoke_assignments": assignments,
            "hub_cost": hub_cost,
            "transportation_cost": transportation_cost,
            "total_cost": hub_cost + transportation_cost,
            "quantum_energy": quantum_result.best_energy,
            "execution_time": quantum_result.execution_time,
            "exactly_one_assignment": exactly_one_assignment,
            "assignments_use_open_hubs": assignments_use_open_hubs,
            "hub_limit_satisfied": hub_limit_satisfied,
            "feasible": feasible,
        }


class QuantumCircuitSimulator:
    """Provide a classical random-bit baseline, not a quantum circuit simulator."""

    def execute_qaoa(
        self, qubo_problem: QUBOProblem, qaoa_circuit: QAOACircuit
    ) -> QuantumResult:
        """Sample random bit strings and retain the lowest-energy sample."""
        if qaoa_circuit.num_qubits != qubo_problem.num_variables:
            raise ValueError("The bit-string length must match the QUBO variable count.")

        rows, columns = np.nonzero(qubo_problem.quadratic_terms)
        coefficients = qubo_problem.quadratic_terms[rows, columns]
        solution_counts: Dict[str, int] = {}
        best_energy = float("inf")
        best_solution = None

        for _ in range(qaoa_circuit.measurement_shots):
            solution = np.random.randint(0, 2, qubo_problem.num_variables)
            energy = self._calculate_qubo_energy(
                solution, qubo_problem, rows, columns, coefficients
            )
            solution_string = "".join(map(str, solution))
            solution_counts[solution_string] = solution_counts.get(solution_string, 0) + 1
            if energy < best_energy:
                best_energy = energy
                best_solution = solution.copy()

        return QuantumResult(
            best_solution=best_solution,
            best_energy=best_energy,
            solution_counts=solution_counts,
            expectation_values=[],
            parameters=(qaoa_circuit.gamma_params.copy(), qaoa_circuit.beta_params.copy()),
            quantum_circuit_depth=0,
            execution_time=0.0,
            convergence_history=[],
            measurement_shots=qaoa_circuit.measurement_shots,
        )

    @staticmethod
    def _calculate_qubo_energy(
        solution: np.ndarray,
        qubo_problem: QUBOProblem,
        rows: np.ndarray,
        columns: np.ndarray,
        coefficients: np.ndarray,
    ) -> float:
        """Evaluate c + q.T @ z + z.T @ Q @ z for a binary vector z."""
        linear_energy = np.dot(qubo_problem.linear_terms, solution)
        quadratic_energy = np.dot(
            coefficients, solution[rows] * solution[columns]
        )
        return float(qubo_problem.constant_term + linear_energy + quadratic_energy)


class ClassicalParameterOptimizer:
    """Explain the parameter-optimization role without claiming to perform it."""

    @staticmethod
    def optimize_qaoa_parameters(
        qubo_problem: QUBOProblem,
        qaoa_circuit: QAOACircuit,
        quantum_simulator: QuantumCircuitSimulator,
    ) -> Tuple[Tuple[List[float], List[float]], List[float]]:
        """Return the supplied parameters unchanged because no QAOA runs here."""
        del qubo_problem, quantum_simulator
        print(
            "QAOA parameter optimization is not run: this notebook has no QAOA circuit."
        )
        return (
            qaoa_circuit.gamma_params.copy(),
            qaoa_circuit.beta_params.copy(),
        ), []


class QuantumPerformanceTracker:
    """Store explicitly supplied performance records for later inspection."""

    def __init__(self):
        self.performance_history = []

    def record_performance(self, quantum_result, classical_benchmarks):
        """Record values without interpreting a classical sample as quantum data."""
        comparison = {
            "sampled_cost": quantum_result["total_cost"],
            "sampled_time": quantum_result["execution_time"],
            "classical_costs": {
                name: data["cost"] for name, data in classical_benchmarks.items()
            },
            "classical_times": {
                name: data["execution_time"]
                for name, data in classical_benchmarks.items()
            },
        }
        self.performance_history.append(comparison)


print("Initializing the illustrative hub-and-spoke QUBO example...")
quantum_optimizer = QuantumHubSpokeOptimizer(num_nodes=25)

print("\nModel configuration:")
print(f"- Network size: {quantum_optimizer.num_nodes} nodes")
print(f"- QAOA depth setting (not executed): {quantum_optimizer.qaoa_depth}")
print(f"- Random bit strings per sample run: {quantum_optimizer.measurement_shots}")
print(f"- Default maximum hubs: {quantum_optimizer.max_hubs}")
print(f"- Penalty coefficient: {quantum_optimizer.qubo_penalty}")
print("\nSynthetic network data summary:")
print(f"- Total directed demand: {np.sum(quantum_optimizer.demand_matrix):,.0f} units")
positive_distances = quantum_optimizer.distance_matrix[
    quantum_optimizer.distance_matrix > 0
]
print(f"- Mean nonzero distance: {np.mean(positive_distances):,.1f} km")
print(
    f"- Hub-opening cost range: ${np.min(quantum_optimizer.hub_costs):,.0f}"
    f" to ${np.max(quantum_optimizer.hub_costs):,.0f}"
)


In [ ]:
print("\n" + "=" * 64)
print("HUB-AND-SPOKE QUBO FORMULATION AND CLASSICAL RANDOM SAMPLING")
print("=" * 64)

# Step 1: Build the QUBO representation of the network model.
start_time = time.perf_counter()
qubo_problem = quantum_optimizer.formulate_qubo(max_hubs=6)
qubo_time = time.perf_counter() - start_time
print(f"\nQUBO construction time: {qubo_time:.3f} seconds")

# Step 2: Store the illustrative circuit settings. No circuit is constructed or run.
qaoa_circuit = QAOACircuit(
    num_qubits=qubo_problem.num_variables,
    depth=quantum_optimizer.qaoa_depth,
    gamma_params=[random.uniform(0, 2 * np.pi) for _ in range(quantum_optimizer.qaoa_depth)],
    beta_params=[random.uniform(0, np.pi) for _ in range(quantum_optimizer.qaoa_depth)],
    measurement_shots=quantum_optimizer.measurement_shots,
)
print("\nStored QAOA settings (not executed):")
print(f"- QUBO binary variables: {qaoa_circuit.num_qubits}")
print(f"- Requested depth: {qaoa_circuit.depth}")
print(f"- Requested samples: {qaoa_circuit.measurement_shots}")

# Step 3: Keep the parameters unchanged; there is no circuit objective to optimize.
optimized_params, convergence_history = (
    quantum_optimizer.classical_optimizer.optimize_qaoa_parameters(
        qubo_problem,
        qaoa_circuit,
        quantum_optimizer.quantum_simulator,
    )
)
qaoa_circuit.gamma_params, qaoa_circuit.beta_params = optimized_params

# Step 4: Sample random binary strings as a classical QUBO-evaluation baseline.
print("\nSampling random binary strings for QUBO evaluation...")
start_time = time.perf_counter()
quantum_result = quantum_optimizer.quantum_simulator.execute_qaoa(
    qubo_problem, qaoa_circuit
)
execution_time = time.perf_counter() - start_time
quantum_result.execution_time = execution_time
quantum_result.convergence_history = convergence_history
print(f"Random-sampling time: {execution_time:.3f} seconds")
print(f"Lowest sampled QUBO energy: {quantum_result.best_energy:,.2f}")
print(f"Distinct sampled bit strings: {len(quantum_result.solution_counts)}")

# Step 5: Decode the selected bit string and check modeled constraints.
quantum_solution = quantum_optimizer._post_process_quantum_result(
    quantum_result, qubo_problem
)
print("\nDecoded sample:")
print(f"- Selected hubs (1-based labels): {[hub + 1 for hub in quantum_solution['hub_configuration']]}")
print(f"- Number of selected hubs: {len(quantum_solution['hub_configuration'])}")
print(f"- Hub-opening cost: ${quantum_solution['hub_cost']:,.2f}")
print(f"- Transportation cost: ${quantum_solution['transportation_cost']:,.2f}")
print(f"- Decoded operating cost: ${quantum_solution['total_cost']:,.2f}")
print(f"- Exactly one assignment per node: {quantum_solution['exactly_one_assignment']}")
print(f"- Every assigned hub is selected: {quantum_solution['assignments_use_open_hubs']}")
print(f"- Hub limit satisfied: {quantum_solution['hub_limit_satisfied']}")
print(f"- Feasible for the modeled constraints: {quantum_solution['feasible']}")
print("- Capacity feasibility: not modeled")

# The decoded cost is the QUBO operating-cost component; penalties affect energy.
assert quantum_result.best_solution.shape == (qubo_problem.num_variables,)
assert len(qubo_problem.variable_labels) == qubo_problem.num_variables
assert np.allclose(qubo_problem.quadratic_terms, qubo_problem.quadratic_terms.T)


In [ ]:
def visualize_quantum_optimization(
    quantum_optimizer: QuantumHubSpokeOptimizer,
    quantum_solution: Dict[str, object],
    quantum_result: QuantumResult,
    qubo_problem: QUBOProblem,
):
    """Plot QUBO sample diagnostics and the decoded network configuration."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Hub-and-Spoke QUBO: Classical Random-Sampling Diagnostics",
        fontsize=16,
        fontweight="bold",
    )

    # The search-space count is shown on a logarithmic scale to avoid overflow.
    ax = axes[0, 0]
    search_space_log10 = qubo_problem.num_variables * math.log10(2)
    metrics = {
        "Binary variables": qubo_problem.num_variables,
        "Random samples": quantum_result.measurement_shots,
        "log10(search-space size)": search_space_log10,
    }
    names = list(metrics)
    values = list(metrics.values())
    bars = ax.bar(names, values, color="#6BAED6", alpha=0.85)
    ax.set_title("Problem and Sampling Scale")
    ax.set_ylabel("Displayed value (search space uses log10)")
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:,.1f}",
            ha="center",
            va="bottom",
        )

    # No convergence curve is shown because no parameter optimization occurred.
    ax = axes[0, 1]
    ax.axis("off")
    ax.set_title("QAOA Parameter Optimization")
    ax.text(
        0.5,
        0.55,
        "Not executed\n\nNo QAOA circuit or parameter-dependent objective is implemented.",
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=12,
        bbox={"boxstyle": "round", "facecolor": "#F0F0F0", "alpha": 0.9},
    )

    # Longitude and latitude are plotted for illustration, not as a map projection.
    ax = axes[1, 0]
    hubs = set(quantum_solution["hub_configuration"])
    for node, (longitude, latitude) in enumerate(quantum_optimizer.coordinates):
        is_hub = node in hubs
        ax.scatter(
            longitude,
            latitude,
            s=180 if is_hub else 60,
            marker="s" if is_hub else "o",
            color="#D73027" if is_hub else "#74ADD1",
            edgecolors="black",
            label="Selected hub" if is_hub and node == min(hubs, default=-1) else (
                "Other node" if not is_hub and node == 0 else None
            ),
            zorder=3,
        )
        ax.annotate(str(node + 1), (longitude, latitude), xytext=(3, 3),
                    textcoords="offset points", fontsize=8)

    assignments = quantum_solution["spoke_assignments"]
    for node, assigned_hubs in assignments.items():
        for hub in assigned_hubs:
            if node != hub:
                node_x, node_y = quantum_optimizer.coordinates[node]
                hub_x, hub_y = quantum_optimizer.coordinates[hub]
                ax.plot([node_x, hub_x], [node_y, hub_y], "--", color="gray", alpha=0.35)
    ax.set_title("Decoded Sample (Assignments Shown as Dashed Lines)")
    ax.set_xlabel("Longitude (degrees)")
    ax.set_ylabel("Latitude (degrees)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    # Counts show the most frequently observed random bit strings, not QAOA outcomes.
    ax = axes[1, 1]
    top_solutions = sorted(
        quantum_result.solution_counts.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:10]
    if top_solutions:
        labels = [f"Sample {i + 1}" for i in range(len(top_solutions))]
        counts = [count for _, count in top_solutions]
        ax.bar(labels, counts, color="#4DAF9C", alpha=0.85)
        ax.set_ylabel("Occurrences")
        ax.set_xlabel("Distinct bit string")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.3)
    ax.set_title("Most Frequent Random Samples")

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_quantum_optimization(
    quantum_optimizer,
    quantum_solution,
    quantum_result,
    qubo_problem,
)


In [ ]:
def analyze_quantum_advantage(
    quantum_optimizer: QuantumHubSpokeOptimizer,
    quantum_solution: Dict[str, object],
    quantum_result: QuantumResult,
    qubo_problem: QUBOProblem,
) -> Dict[str, object]:
    """Summarize model size and sample feasibility without claiming advantage."""
    log10_search_space = qubo_problem.num_variables * math.log10(2)
    sample_fraction = quantum_result.measurement_shots / (2 ** qubo_problem.num_variables)

    print("\n=== MODEL AND SAMPLING ANALYSIS ===")
    print("Sample quality:")
    print(f"- Lowest sampled QUBO energy: {quantum_result.best_energy:,.2f}")
    print(f"- Decoded operating cost: ${quantum_solution['total_cost']:,.2f}")
    print(f"- Modeled feasibility: {quantum_solution['feasible']}")
    print("- Capacity constraints: not included")

    print("\nModel scale:")
    print(f"- Binary variables: {qubo_problem.num_variables}")
    print(f"- Search-space size: approximately 10^{log10_search_space:.1f} bit strings")
    print(f"- Random samples drawn: {quantum_result.measurement_shots}")
    print(f"- Fraction of search space sampled: approximately {sample_fraction:.3e}")
    print("- Quantum circuit executions: 0")

    print("\nInterpretation:")
    print(
        "The sampler evaluates a small random subset of bit strings. Its lowest-energy "
        "sample is not a proof of optimality, and it may violate the modeled constraints."
    )
    print(
        "No quantum-versus-classical quality or runtime comparison is available from this run."
    )
    print(
        "The penalty coefficient is finite; feasibility must be checked rather than inferred "
        "from a low penalized energy."
    )
    print(
        "A genuine QAOA study would require a circuit implementation, a parameter-dependent "
        "expectation estimate, and a clearly specified classical benchmark."
    )

    return {
        "num_variables": qubo_problem.num_variables,
        "log10_search_space": log10_search_space,
        "sample_fraction": sample_fraction,
        "sampled_energy": quantum_result.best_energy,
        "decoded_cost": quantum_solution["total_cost"],
        "modeled_feasible": quantum_solution["feasible"],
        "quantum_circuit_executions": 0,
        "execution_time": quantum_result.execution_time,
    }


advantage_analysis = analyze_quantum_advantage(
    quantum_optimizer,
    quantum_solution,
    quantum_result,
    qubo_problem,
)


## The QAOA workflow in context

QAOA is a hybrid quantum-classical approach. For a binary optimization objective represented by a diagonal problem Hamiltonian $H_C$, a QAOA state at depth $p$ has the form

$$|\boldsymbol{\gamma},\boldsymbol{\beta}\rangle
=U_M(\beta_p)U_C(\gamma_p)\cdots U_M(\beta_1)U_C(\gamma_1)|+\rangle^{\otimes m},$$

where $m$ is the number of binary variables, $|+\rangle^{\otimes m}$ is an equal superposition over computational-basis bit strings, and $U_C(\gamma)=e^{-i\gamma H_C}$ and $U_M(\beta)=e^{-i\beta H_M}$ are problem and mixer unitaries. Measuring the state produces candidate bit strings. A classical optimizer typically updates $\boldsymbol{\gamma}$ and $\boldsymbol{\beta}$ using estimated objective values from repeated circuit measurements.

The mathematical expression describes the QAOA method, not an operation performed by this notebook. Here, the `QAOACircuit` dataclass stores illustrative settings, while `QuantumCircuitSimulator.execute_qaoa` draws independent random bits and evaluates their QUBO energies. The parameters do not affect those samples, there is no quantum state or circuit, and no parameter convergence curve exists. This distinction is essential when interpreting any runtime or solution-quality result.

### Strengths and limitations of the model

The QUBO representation makes binary decisions and selected constraints explicit in a single energy function, which is useful when studying mappings to Ising models or quantum optimization circuits. The slack-variable construction represents an at-most hub limit using a quadratic penalty while retaining the same hub-selection decision.

The formulation also has important limitations. The finite penalty value may be too small relative to the operating-cost coefficients to prioritize feasibility. Capacity limits are not enforced, and the transportation objective includes only demand-weighted distance from assigned nodes to their hubs. The current sampler does not optimize the QUBO; it is only a random reference procedure. Consequently, this example supports learning about QUBO modeling and feasibility checks, but it does not demonstrate quantum speedup, quantum advantage, QAOA approximation guarantees, or production readiness.

### Appropriate interpretation

Use the formulation to trace how hub decisions, assignments, slack bits, and penalties become coefficients in a quadratic binary objective. Use the sampled result to practice decoding a bit string and checking model constraints. Treat any observed cost as the cost of that sampled configuration, not as an optimum. A complete operational model would need additional assumptions and constraints—such as hub capacities, service requirements, and any hub-to-hub transportation legs—specified before constructing and evaluating its QUBO.